# CS-317: Reinforcement Learning — Lab 7
## Model-Free Prediction with Monte Carlo Methods
**BS AI-1**

This notebook provides the corridor environment, the two policies, the exact reference values, the FrozenLake environment and plotting helpers. Complete every `# TODO` cell, run all cells top to bottom without errors, and answer the written questions in the markdown cells provided for them.

- Set your personal **SEED** (the last three digits of your roll number) in the first code cell below. Every random experiment must use it, and your written answers must refer to your own numbers.
- **Predict first** cells must be filled in *before* you run the code beneath them.
- The exact values (`V_exact_R`, `V_exact_U`, `V_true_U`, `V_exact_D`, `V_exact_C`) are for **checking only**. No estimate may be computed from them, and your Monte Carlo code must never use `P` or `R`.

Reinforcement-learning libraries may not be used; Gymnasium may be used only to create the FrozenLake environment of Part 5 (in Colab, run `!pip install gymnasium` if it is missing).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

In [1]:
# ---------------------------------------------------------------
# Personal seed: the last three digits of your roll number.
# Every random experiment in this notebook must use it.
# ---------------------------------------------------------------
SEED = 808          # TODO: e.g. SEED = 417

assert SEED is not None, "Set SEED to the last three digits of your roll number before running the notebook."
print("SEED =", SEED)

SEED = 808


---
## Provided Starter Code

Run the cells in this section as-is. You do not need to modify them.

| Name | What it gives you |
|---|---|
| `CorridorEnv(p_move=0.8, seed=None)` | `reset()` → `s` ; `step(a)` → `s2, r, done`. The model is hidden. |
| `right_policy`, `random_policy` | arrays of shape (5, 2) holding $\pi(a\mid s)$ |
| `run_episode(env, policy, rng)` | → `states, actions, rewards` (T entries each; `rewards[t]` is $R_{t+1}$) |
| `V_exact_R`, `V_exact_U` | exact values from Lab 5 (checking only) |
| `plot_values(V, V_ref=None, title)` | bar plot of estimates against a reference |
| `build_corridor(p_move)` | `P[s,a,s2]`, `R[s,a,s2]` as in Lab 5 (**Task 5 only**) |
| `V_true_U` | exact values of $\pi_U$ for `p_move = 0.6` (Task 5 checking) |
| `make_frozenlake(seed)` | 4×4 slippery FrozenLake, `reset()` / `step()` only |
| `pi_direct`, `pi_cautious` | arrays of shape (16, 4) holding $\pi(a\mid s)$ |
| `V_exact_D`, `V_exact_C` | exact FrozenLake values (checking only) |
| `plot_frozenlake(V, title)` | 4×4 heat map with holes and goal marked |

In [ ]:
# ---------------------------------------------------------------
# Warehouse corridor (same dynamics as Lab 5)
#
#   cell:   0     1     2     3     4
#           F     .     S     .     G
#
#   Actions: 0 = Left, 1 = Right
# ---------------------------------------------------------------

N_STATES, N_ACTIONS = 5, 2
ACTIONS = ["Left", "Right"]
TERMINALS = (0, 4)
START = 2
GAMMA = 0.9


def _env_stream_seed(seed, salt):
    """Derives the environment's own seed from `seed`, so that CorridorEnv(seed=k) and
    np.random.default_rng(k) produce INDEPENDENT random streams (otherwise the policy's
    coin flips and the environment's slips would be the same numbers)."""
    if seed is None:
        return None
    return int(np.random.SeedSequence([int(seed), salt]).generate_state(1)[0])


class CorridorEnv:
    """The corridor as an environment you can only ACT in.

    reset()       -> s                 the start cell (2)
    step(action)  -> s2, r, done       next cell, reward, end-of-episode flag

    The slip probability and the arrays P and R are hidden: reading env.P or
    env.R raises an error, exactly as in a real system whose rules you do not know.
    """

    def __init__(self, p_move=0.8, seed=None):
        self.__p_move = p_move                       # hidden dynamics
        self.__rng = np.random.default_rng(_env_stream_seed(seed, 101))
        self.__s = None

    def __getattr__(self, name):
        if name in ("P", "R", "p_move", "transition_matrix"):
            raise AttributeError(f"The model is hidden: '{name}' is not available. "
                                 "This environment only offers reset() and step(action).")
        raise AttributeError(name)

    def reset(self):
        self.__s = START
        return self.__s

    def step(self, action):
        if self.__s is None or self.__s in TERMINALS:
            raise RuntimeError("Episode has ended (or not started): call reset() first.")
        s = self.__s
        if self.__rng.random() < self.__p_move:      # intended move
            s2 = s + (-1 if action == 0 else +1)
        else:                                        # slip: stay in place
            s2 = s
        r = 10.0 if s2 == 4 else (-10.0 if s2 == 0 else -1.0)
        self.__s = s2
        return s2, r, s2 in TERMINALS


# The two policies of Table 3, as pi[s, a] (rows of the terminal cells are never used)
right_policy = np.zeros((N_STATES, N_ACTIONS)); right_policy[:, 1] = 1.0     # pi_R
random_policy = np.full((N_STATES, N_ACTIONS), 0.5)                           # pi_U


def run_episode(env, policy, rng, max_steps=10_000):
    """Plays one episode from env.reset() following `policy` (shape (n_states, n_actions)).

    Returns three lists with T entries each:
      states  = [S0, S1, ..., S_{T-1}]   (the terminal state is NOT included)
      actions = [A0, A1, ..., A_{T-1}]
      rewards = [R1, R2, ..., R_T]       (rewards[t] is the reward that FOLLOWS action t)
    """
    s = env.reset()
    states, actions, rewards = [], [], []
    for _ in range(max_steps):
        a = int(rng.choice(policy.shape[1], p=policy[s]))
        s2, r, done = env.step(a)
        states.append(int(s)); actions.append(a); rewards.append(float(r))
        s = s2
        if done:
            break
    return states, actions, rewards

In [ ]:
# ---------------------------------------------------------------
# Planning model and reference values.
# build_corridor is the Lab 5 model; use it ONLY in Task 5.
# The exact values are for CHECKING your estimates only: no estimate
# may be computed from them.
# ---------------------------------------------------------------

def build_corridor(p_move=0.8, step=-1.0, goal=10.0, forklift=-10.0):
    """Lab 5 model: P[s, a, s2] and R[s, a, s2]. Terminal cells are absorbing with reward 0."""
    P = np.zeros((N_STATES, N_ACTIONS, N_STATES))
    R = np.zeros((N_STATES, N_ACTIONS, N_STATES))
    for t in TERMINALS:
        P[t, :, t] = 1.0
    for s in [1, 2, 3]:
        for a, d in enumerate([-1, +1]):
            s2 = s + d
            r = goal if s2 == 4 else (forklift if s2 == 0 else step)
            P[s, a, s2] += p_move;     R[s, a, s2] = r
            P[s, a, s] += 1 - p_move;  R[s, a, s] = step
    return P, R


def _exact_values(P, R, policy, gamma):
    P_pi = np.einsum("sa,sat->st", policy, P)
    r_pi = np.einsum("sa,sat,sat->s", policy, P, R)
    V = np.linalg.solve(np.eye(P.shape[0]) - gamma * P_pi, r_pi)
    V[list(TERMINALS)] = 0.0
    return V


_P08, _R08 = build_corridor(0.8)
_P06, _R06 = build_corridor(0.6)
V_exact_R = _exact_values(_P08, _R08, right_policy, GAMMA) + 0.0    # pi_R, real corridor p = 0.8
V_exact_U = _exact_values(_P08, _R08, random_policy, GAMMA) + 0.0   # pi_U, real corridor p = 0.8
V_true_U = _exact_values(_P06, _R06, random_policy, GAMMA) + 0.0    # pi_U, real corridor p = 0.6 (Task 5)
del _P08, _R08, _P06, _R06

print("V_exact_R =", V_exact_R)
print("V_exact_U =", V_exact_U)


def plot_values(V, V_ref=None, title="", cells=(1, 2, 3)):
    """Bar plot of the estimates V for the non-terminal cells, next to a reference
    (e.g. V_exact_U) if one is given. Cells with no estimate (np.nan) are marked '–'."""
    cells = list(cells)
    x = np.arange(len(cells))
    est = np.array([V[c] for c in cells], dtype=float)
    fig, ax = plt.subplots(figsize=(6, 3.2))
    w = 0.38 if V_ref is not None else 0.6
    ax.bar(x - (w / 2 if V_ref is not None else 0), np.nan_to_num(est), w, label="estimate", color="#4C72B0")
    if V_ref is not None:
        ax.bar(x + w / 2, [V_ref[c] for c in cells], w, label="exact", color="#BBBBBB")
    for i, v in enumerate(est):
        ax.text(x[i] - (w / 2 if V_ref is not None else 0), 0 if np.isnan(v) else v,
                "–" if np.isnan(v) else f"{v:.2f}", ha="center",
                va="bottom" if (np.isnan(v) or v >= 0) else "top", fontsize=9)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(x); ax.set_xticklabels([f"cell {c}" for c in cells])
    ax.set_ylabel("V(s)"); ax.set_title(title)
    if V_ref is not None:
        ax.legend()
    plt.tight_layout(); plt.show()

In [ ]:
# ---------------------------------------------------------------
# FrozenLake 4x4 (slippery), used in Part 5 (Task 6)
#
#   S F F F        cells numbered 0..15 row by row
#   F H F H        holes: 5, 7, 11, 12     goal: 15
#   F F F H
#   H F F G        actions: 0 = Left, 1 = Down, 2 = Right, 3 = Up
# ---------------------------------------------------------------

FL_MAP = ["SFFF", "FHFH", "FFFH", "HFFG"]
FL_ACTIONS = ["Left", "Down", "Right", "Up"]
FL_N_STATES, FL_N_ACTIONS = 16, 4
FL_HOLES, FL_GOAL = (5, 7, 11, 12), 15
FL_TERMINALS = FL_HOLES + (FL_GOAL,)

try:
    import gymnasium as gym
    _HAVE_GYM = True
except ImportError:                     # in Colab: !pip install gymnasium
    _HAVE_GYM = False
    print("gymnasium not found: using an identical built-in FrozenLake (same map and slip rule).")


def _fl_move(s, a):
    r, c = divmod(s, 4)
    if a == 0: c = max(c - 1, 0)
    elif a == 1: r = min(r + 1, 3)
    elif a == 2: c = min(c + 1, 3)
    else: r = max(r - 1, 0)
    return r * 4 + c


class _FrozenLakeHidden:
    """FrozenLake that, like CorridorEnv, only offers reset() and step(action)."""

    def __init__(self, seed=None):
        self.__seed = _env_stream_seed(seed, 202)
        self.__first = True
        if _HAVE_GYM:
            # .unwrapped removes Gymnasium's default 100-step time limit, so that an
            # episode only ends at the goal or in a hole (as assumed by the exact values).
            self.__env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True).unwrapped
        else:
            self.__rng = np.random.default_rng(self.__seed)
        self.__s = None

    def __getattr__(self, name):
        if name in ("P", "R", "unwrapped", "transition_matrix"):
            raise AttributeError(f"The model is hidden: '{name}' is not available. "
                                 "This environment only offers reset() and step(action).")
        raise AttributeError(name)

    def reset(self):
        if _HAVE_GYM:
            s, _ = self.__env.reset(seed=self.__seed if self.__first else None)
        else:
            s = 0
        self.__first = False
        self.__s = int(s)
        return self.__s

    def step(self, action):
        if self.__s is None or self.__s in FL_TERMINALS:
            raise RuntimeError("Episode has ended (or not started): call reset() first.")
        if _HAVE_GYM:
            s2, r, terminated, truncated, _ = self.__env.step(int(action))
            s2, done = int(s2), bool(terminated or truncated)
        else:
            a = (int(action) + self.__rng.integers(-1, 2)) % 4      # intended or one of the two sideways
            s2 = _fl_move(self.__s, a)
            r, done = (1.0 if s2 == FL_GOAL else 0.0), s2 in FL_TERMINALS
        self.__s = s2
        return s2, float(r), done


def make_frozenlake(seed=None):
    """4x4 slippery FrozenLake with the model hidden: reset() -> s ; step(a) -> s2, r, done."""
    return _FrozenLakeHidden(seed)


# Policies of Table 14, as pi[s, a] of shape (16, 4).
# Rows of holes and the goal are never used.
_direct = [1 if s % 4 == 3 else 2 for s in range(FL_N_STATES)]              # Right; Down in the right column
_cautious = [0, 3, 3, 3, 0, 0, 0, 0, 3, 1, 0, 0, 0, 2, 1, 0]
pi_direct = np.eye(FL_N_ACTIONS)[_direct]
pi_cautious = np.eye(FL_N_ACTIONS)[_cautious]


def _fl_exact(policy):
    """Exact values (gamma = 1) from the hidden model; for checking only."""
    P = np.zeros((FL_N_STATES, FL_N_STATES)); r = np.zeros(FL_N_STATES)
    for s in range(FL_N_STATES):
        if s in FL_TERMINALS:
            continue
        for a in range(FL_N_ACTIONS):
            for d in (-1, 0, 1):
                s2 = _fl_move(s, (a + d) % 4)
                p = policy[s, a] / 3
                if s2 == FL_GOAL: r[s] += p
                if s2 not in FL_TERMINALS: P[s, s2] += p
    return np.linalg.solve(np.eye(FL_N_STATES) - P, r)


V_exact_D = _fl_exact(pi_direct)       # pi_D: probability of reaching the goal from each cell
V_exact_C = _fl_exact(pi_cautious)     # pi_C


def plot_frozenlake(V, title=""):
    """4x4 heat map of V with holes (H) and goal (G) marked. Cells with no estimate (np.nan) show '–'."""
    V = np.asarray(V, dtype=float)
    grid = V.reshape(4, 4)
    fig, ax = plt.subplots(figsize=(4.2, 4.2))
    ax.imshow(np.nan_to_num(grid), cmap="Blues", vmin=0, vmax=1)
    for s in range(FL_N_STATES):
        r, c = divmod(s, 4)
        if s in FL_HOLES:
            ax.add_patch(plt.Rectangle((c - .5, r - .5), 1, 1, color="#444444"))
            ax.text(c, r, "H", ha="center", va="center", color="white", fontsize=13, fontweight="bold")
        elif s == FL_GOAL:
            ax.add_patch(plt.Rectangle((c - .5, r - .5), 1, 1, color="#8FD18F"))
            ax.text(c, r, "G", ha="center", va="center", fontsize=13, fontweight="bold")
        else:
            v = grid[r, c]
            ax.text(c, r, "–" if np.isnan(v) else f"{v:.2f}", ha="center", va="center",
                    color="white" if (not np.isnan(v) and v > 0.6) else "black", fontsize=11)
            ax.text(c - .42, r - .32, str(s), fontsize=7,
                    color="white" if (not np.isnan(v) and v > 0.6) else "dimgray")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title)
    plt.tight_layout(); plt.show()


print("Exact V(0):  pi_D =", round(V_exact_D[0], 3), "  pi_C =", round(V_exact_C[0], 3))

---
## Part 1: Model-Based and Model-Free Methods

### Task 1: Lock the Model and Classify Methods  *(15 min)*

**(a)** Create `env = CorridorEnv(seed=SEED)`. Try to read `env.P` and `env.R` and record what happens. Then generate one episode under $\pi_U$ with `run_episode` and print its states, actions and rewards.

In [ ]:
env = CorridorEnv(seed=SEED)
rng = np.random.default_rng(SEED)

# TODO: try to read env.P and env.R (use try/except so the notebook keeps running)
#       and print the error message.
# your code here


# TODO: generate one episode under random_policy with run_episode and print
#       its states, actions (by name) and rewards.
# your code here

**(b)** Table 5: Which approach fits which situation?

| Scenario | Model known? | Model-based or model-free? | Justification |
|---|---|---|---|
| The corridor of Lab 5, with P and R printed in the manual | | | |
| The same corridor, but the slip probability is unknown and only a working simulator is available | | | |
| A slippery ice-grid game (FrozenLake) that you can play but whose rules are hidden | | | |
| A chess program that knows all the rules and can look ahead through every legal move | | | |
| A delivery robot whose wheels wear out, so its dynamics drift from week to week | | | |



**Questions**

1. What happened when you tried to read `env.P`? Name the line of your Lab 5 policy evaluation code that can no longer run, and say what it needs.

   *your answer here*

2. In the chess scenario the model is known exactly, yet exact dynamic programming is not used. What stops it? What does this show about the statement "model known, therefore use dynamic programming"?

   *your answer here*

3. In the second scenario the probabilities are unknown but a simulator exists. Can Monte Carlo use it? Can dynamic programming? State exactly what each one needs from the simulator.

   *your answer here*

---
## Part 2: Returns and Visits

### Task 2: Returns and Visits by Hand  *(20 min)*

The return of every time step is computed backward from the end of the episode:

$$G_T = 0, \qquad G_t = R_{t+1} + \gamma\, G_{t+1}$$

Under **first-visit** MC only the first occurrence of a state in the episode contributes a return; under **every-visit** MC every occurrence contributes one.

**Table 7: Episode E** (generated under $\pi_U$; the reward in row $t$ is $R_{t+1}$; $G_5 = 0$; $\gamma = 0.9$)

| $t$ | $S_t$ | $A_t$ | $R_{t+1}$ | $G_t$ |
|---|---|---|---|---|
| 0 | 2 | Right | –1 | |
| 1 | 2 | Right | –1 | |
| 2 | 3 | Left | –1 | |
| 3 | 2 | Right | –1 | |
| 4 | 3 | Right | +10 | |

**(a)** Complete the column $G_t$ by hand, working from $t = 4$ back to $t = 0$.

*your working here*

**Predict first.** Will the every-visit estimate for cell 3 from this episode be larger or smaller than the first-visit estimate, and why?

*your prediction here*

**(b)** List the cells that are visited more than once and the time steps of every visit.

*your answer here*

**(c)** Table 8: Estimates from episode E alone (use only the returns from part (a))

| Cell | First-visit returns used | First-visit estimate | Every-visit returns used | Every-visit estimate |
|---|---|---|---|---|
| 2 | | | | |
| 3 | | | | |

**(d)** Implement `returns_backward(rewards, gamma)` with a single backward loop and check it against your hand values.

In [ ]:
# Episode E from Table 7
E_states  = [2, 2, 3, 2, 3]
E_actions = [1, 1, 0, 1, 1]          # 0 = Left, 1 = Right
E_rewards = [-1, -1, -1, -1, 10]     # E_rewards[t] = R_{t+1}


def returns_backward(rewards, gamma):
    # Returns a list G with G[t] = R_{t+1} + gamma * G[t+1] and G_T = 0.
    # TODO: one backward loop, from t = T-1 down to 0
    # your code here

    return G


# TODO: print returns_backward(E_rewards, GAMMA) and compare with your hand values.
# your code here

**(e)** A classmate computes the returns with the forward loop below and says it gives the same $G_t$. Run it on episode E and report the values it returns for $t = 0$ and $t = 4$, then explain at one of these time steps what the loop is actually computing.

In [ ]:
def returns_forward(rewards, gamma):
    # The classmate's version (do not modify)
    T = len(rewards)
    returns = [0.0] * T
    G = 0
    for t in range(T):               # forward loop
        G = rewards[t] + gamma * G
        returns[t] = G
    return returns


# TODO: run returns_forward on E_rewards and compare with returns_backward.
# your code here

*your explanation here*

**Questions**

1. In episode E, which of $G_0, \dots, G_4$ would change if the first reward were –5 instead of –1, and which would not? Explain using the backward formula.

   *your answer here*

2. The every-visit returns of cell 2 in episode E are $G_0$, $G_1$ and $G_3$. Express $G_0$ in terms of $G_1$. What does this say about treating the three returns as independent samples?

   *your answer here*

3. Does your prediction about cell 3 match Table 8? If not, find where your reasoning went wrong.

   *your answer here*

---
## Part 3: Monte Carlo Prediction

First-visit MC keeps, for every state, a count $N(s)$ of the returns collected and their sum $S(s)$; the estimate is $V(s) = S(s)/N(s)$.

```
initialise N(s) = 0, S(s) = 0 for all states
repeat for every episode:
    generate an episode S0, A0, R1, ..., ST by following policy pi
    G = 0
    for t = T-1, T-2, ..., 0:
        G = R(t+1) + gamma * G
        if S(t) does not appear in S0, ..., S(t-1):        # first-visit check
            N(S(t)) += 1 ;  S(S(t)) += G ;  V(S(t)) = S(S(t)) / N(S(t))
```

### Task 3: First-Visit Monte Carlo Prediction  *(30 min)*

Implement `mc_first_visit(env, policy, n_episodes, gamma, rng)`. It must return `V`, `N` and a record of `V(2)` after every episode, and it must use the running sums `S` and `N` rather than storing lists of returns. Use $\gamma = 0.9$.

In [ ]:
def mc_first_visit(env, policy, n_episodes, gamma, rng):
    # Returns:
    #   V          : array of shape (n_states,), np.nan for states with N(s) = 0
    #   N          : array of shape (n_states,), number of returns used for each state
    #   V2_history : array of shape (n_episodes,), the estimate V(2) after every episode
    # Only env.reset(), env.step() (through run_episode) and the policy may be used.
    n_states = policy.shape[0]
    N = np.zeros(n_states)
    S = np.zeros(n_states)
    V2_history = np.full(n_episodes, np.nan)
    # TODO
    # your code here

    V = np.divide(S, N, out=np.full(n_states, np.nan), where=N > 0)
    return V, N, V2_history

**Predict first.** For $\pi_R$, what do you expect $N(1)$ and $N(3)$ to be after 10,000 episodes? Which cell do you expect to have the smallest $N(s)$ under $\pi_U$?

*your prediction here*

**(a)** Run the function for $\pi_R$ and $\pi_U$ with 10, 100, 1000 and 10,000 episodes (a new `CorridorEnv(seed=SEED)` and `np.random.default_rng(SEED)` for every run), and complete Table 9 for the start cell.

In [ ]:
# TODO: Table 9 — V(2) and its absolute error after 10, 100, 1000 and 10,000 episodes
#       for right_policy and random_policy.
# your code here

| Episodes | V(2) under $\pi_R$ | Absolute error | V(2) under $\pi_U$ | Absolute error |
|---|---|---|---|---|
| 10 | | | | |
| 100 | | | | |
| 1000 | | | | |
| 10,000 | | | | |

**(b)** For the 10,000-episode run, complete Table 10 for all non-terminal cells. Write "–" where a cell has no estimate. `plot_values` may help.

In [ ]:
# TODO: Table 10 — N(s), estimate and exact value for cells 1, 2, 3 under both policies
#       (10,000 episodes). Plot each estimate against its exact values with plot_values.
# your code here

| Cell | $\pi_R$: N(s) | $\pi_R$: estimate | $\pi_R$: exact | $\pi_U$: N(s) | $\pi_U$: estimate | $\pi_U$: exact |
|---|---|---|---|---|---|---|
| 1 | | | | | | |
| 2 | | | | | | |
| 3 | | | | | | |

**(c)** For $\pi_U$, run five independent experiments (seeds SEED, SEED+1, …, SEED+4) of 10,000 episodes. Plot $|V(2) - V_\pi(2)|$ against the number of episodes for all five on one log-log plot, and add a reference line proportional to $n^{-1/2}$.

In [ ]:
# TODO: five runs (env and rng both seeded with SEED + k), one log-log plot with the
#       five error curves and a reference line c / sqrt(n). Label axes, add a legend.
# your code here

**Questions**

1. Under $\pi_R$ your code has no estimate for cell 1, yet Lab 5 gave a value for it. Is this a bug? Which question does the Lab 5 value answer, and why can episodes that always start in cell 2 never answer it?

   *your answer here*

2. Under first-visit MC, $N(2)$ equals the number of episodes exactly. Why? What would $N(2)$ be under every-visit MC, and why would it be larger?

   *your answer here*

3. Read your own plot. By what factor did the error shrink between 100 and 10,000 episodes in each of your five runs? Is that what $n^{-1/2}$ predicts? Explain any run that does not follow the line.

   *your answer here*

4. After 10 episodes your estimate of V(2) is far from the exact value. Is the estimate wrong, or only noisy? State the difference and say how you could tell from your five runs.

   *your answer here*

### Task 4: Every-Visit Monte Carlo and the Bias-Variance Trade-Off  *(25 min)*

Obtain `mc_every_visit` by changing only the first-visit check of your Task 3 function. Both estimators must be computed from the **same** episodes in every run, so that any difference comes from the estimator and not from the sampling.

**Hint:** an easy way to guarantee this is to generate the episodes of a run once (a list of `(states, actions, rewards)`) and pass the same list to a function that returns both estimates.

In [ ]:
def mc_every_visit(env, policy, n_episodes, gamma, rng):
    # Same as mc_first_visit, with only the first-visit check changed.
    # TODO
    # your code here

    pass


def mc_both_from_episodes(episodes, n_states, gamma):
    # Given a list of episodes [(states, actions, rewards), ...], returns
    # (V_first, V_every): the first-visit and every-visit estimates from the SAME episodes
    # (np.nan where a state has no return).
    # TODO
    # your code here

    pass

**(a)** Run 2000 independent runs of 5 episodes each (run $k$ uses seed SEED + $k$) for $\pi_U$ and $\pi_R$. For every run store the first-visit and the every-visit estimate of cells 2 and 3. Complete Table 11 with the mean, the bias (mean minus exact value), the standard deviation across runs, and the standard error of the mean (std. dev. / $\sqrt{\text{number of runs}}$).

In [ ]:
# TODO: Table 11 — 2000 runs x 5 episodes, both policies, cells 2 and 3, both estimators.
# your code here

| Policy | Cell | Estimator | Mean estimate | Exact | Bias | Std. dev. across runs | Std. error of the mean |
|---|---|---|---|---|---|---|---|
| $\pi_U$ | 2 | First-visit | | | | | |
| $\pi_U$ | 2 | Every-visit | | | | | |
| $\pi_U$ | 3 | First-visit | | | | | |
| $\pi_U$ | 3 | Every-visit | | | | | |
| $\pi_R$ | 2 | First-visit | | | | | |
| $\pi_R$ | 2 | Every-visit | | | | | |
| $\pi_R$ | 3 | First-visit | | | | | |
| $\pi_R$ | 3 | Every-visit | | | | | |

**(b)** For $\pi_U$ and cell 3, repeat the experiment with 3, 5, 10, 20 and 50 episodes per run (1000 runs each) and plot the bias of both estimators against the number of episodes per run.

In [ ]:
# TODO: bias of both estimators (pi_U, cell 3) for 3, 5, 10, 20, 50 episodes per run,
#       1000 runs each, on one labelled plot.
# your code here

**Questions**

1. Which estimator shows a bias in your table, for which policy and cell? Is each bias larger than two standard errors? State which entries you would call distinguishable from zero.

   *your answer here*

2. Using your answer to the second question of Task 2, explain in two sentences why the every-visit estimate can be biased when only a few episodes are used.

   *your answer here*

3. Look at your bias plot. What happens to the bias of each estimator as the number of episodes per run grows? What does this say about whether every-visit MC is still a usable estimator?

   *your answer here*

4. A classmate claims that every-visit MC always has the smaller variance. Does your table support the claim for both policies? Quote the numbers you rely on.

   *your answer here*

---
## Part 4: Monte Carlo versus Dynamic Programming

### Task 5: When the Model Is Wrong  *(25 min)*

The real corridor uses $p = 0.6$, while the planning model built by `build_corridor` still uses $p = 0.8$. Evaluate $\pi_U$ in both ways. `V_true_U` may be used **only after** the estimates have been computed.

**(a)** Paste your Lab 5 `policy_evaluation` below and run it on the planning model ($p = 0.8$). Record V for cells 1, 2 and 3.

In [ ]:
P_plan, R_plan = build_corridor(p_move=0.8)      # the planner's (wrong) model

# TODO: paste your Lab 5 policy_evaluation here and evaluate random_policy on
#       (P_plan, R_plan) with gamma = 0.9.
# your code here

**(b)** Create the real environment with `CorridorEnv(p_move=0.6, seed=SEED)` and run first-visit MC for 100, 1000 and 5000 episodes. Also record the total number of environment steps that the 5000-episode run used.

In [ ]:
# TODO: MC on the real environment (p_move = 0.6) for 100, 1000 and 5000 episodes,
#       and the total number of env.step() calls of the 5000-episode run.
# your code here

**(c)** Table 12: Estimates of V under the random policy in the real environment ($p = 0.6$)

| Cell | DP on the planning model ($p = 0.8$) | MC, 100 episodes | MC, 1000 episodes | MC, 5000 episodes | True value ($p = 0.6$) |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**(d)** Plot the absolute error of the MC estimate of V(2) against the number of episodes (use 50 values between 10 and 5000), and draw the absolute error of the dynamic programming result as a horizontal line. Report the smallest number of episodes at which your MC curve stays below that line.

In [ ]:
# TODO: error of the MC estimate of V(2) at 50 episode counts between 10 and 5000
#       (e.g. read them from V2_history of the 5000-episode run), the DP error as a
#       horizontal line, and the crossover point.
# your code here

**Questions**

1. Which method's error is systematic and which is random? Use Table 12 and your plot to show what happens to each error when you spend more computation or more episodes.

   *your answer here*

2. At what number of episodes did MC overtake dynamic programming in your plot? Would a classmate with a different seed necessarily find the same number? Justify your answer from your own five runs of Task 3.

   *your answer here*

3. Someone says "dynamic programming is exact and Monte Carlo is only approximate". Complete the sentence "dynamic programming is exact with respect to ..." and say why that matters here.

   *your answer here*

4. Suppose the real slip probability were 0.78 instead of 0.6. Would you still prefer Monte Carlo? Use the crossover you found to argue for or against.

   *your answer here*

5. Your dynamic programming run used no environment steps and your MC run used thousands. Why are these two costs not directly comparable? In which real application would the MC cost be the more serious one?

   *your answer here*

---
## Part 5: A Model-Free Example, FrozenLake

The walker moves in the chosen direction with probability 1/3 and in each of the two sideways directions with probability 1/3. Reaching the goal gives +1, falling into a hole gives 0, and there is no discounting ($\gamma = 1$). The value of a cell is therefore the **probability of reaching the goal** from that cell under the policy.

### Task 6: Evaluate Two FrozenLake Policies  *(35 min)*

**(a)** Adapt `mc_first_visit` to FrozenLake. Only array sizes and the discount should change; the backward loop and the first-visit check stay as they are. (If your Task 3 function takes its sizes from `policy.shape`, it may work unchanged.)

In [ ]:
fl_env = make_frozenlake(seed=SEED)
print("Start cell:", fl_env.reset())
plot_frozenlake(np.full(FL_N_STATES, np.nan), title="FrozenLake layout")

# TODO: adapt (or reuse) mc_first_visit for FrozenLake: 16 states, gamma = 1.
# your code here

**Predict first.** Write down your guess of V(0) for each policy as a probability, and say which policy you expect to visit cell 14 more often.

*your prediction here*

**(b)** Run 10,000 episodes for each policy and record the mean episode length of each. Complete Table 15 and plot both value functions as 4 × 4 heat maps with `plot_frozenlake`.

In [ ]:
# TODO: 10,000 episodes for pi_direct and pi_cautious (seeded with SEED), mean episode
#       length of each, Table 15, and two plot_frozenlake heat maps.
# your code here

| Cell | $\pi_D$: N(s) | $\pi_D$: estimate | $\pi_D$: exact | $\pi_C$: N(s) | $\pi_C$: estimate | $\pi_C$: exact |
|---|---|---|---|---|---|---|
| 0 | | | | | | |
| 1 | | | | | | |
| 6 | | | | | | |
| 14 | | | | | | |
| Your choice: ___ | | | | | | |

Mean episode length: $\pi_D$ = ___ , $\pi_C$ = ___

**(c)** Every return is 0 or 1, so the standard error of an estimate built from N returns is $\sqrt{V(1-V)/N}$. Report V(0) for both policies with a 95% interval (the estimate ± 1.96 standard errors) and say whether the exact value lies inside it.

In [ ]:
# TODO: 95% interval for V(0) under both policies, and whether the exact value lies inside.
# your code here

**(d)** Repeat the experiment of Task 4 on FrozenLake: 1000 independent runs of 5 episodes (run $k$ uses seed SEED + $k$), with the first-visit and every-visit estimates computed from the same episodes. Complete Table 16.

In [ ]:
# TODO: Table 16 — pi_C cell 0 and pi_D cell 14, both estimators, 1000 runs x 5 episodes.
#       Count the runs in which the cell received no estimate.
# your code here

| Policy | Cell | Estimator | Mean estimate | Exact | Std. dev. across runs | Runs with no estimate |
|---|---|---|---|---|---|---|
| $\pi_C$ | 0 | First-visit | | | | |
| $\pi_C$ | 0 | Every-visit | | | | |
| $\pi_D$ | 14 | First-visit | | | | |
| $\pi_D$ | 14 | Every-visit | | | | |

**Questions**

1. Compare the intervals for V(0) under the two policies. Is the difference between the policies larger than the noise in your estimates?

   *your answer here*

2. In cell 1, $\pi_D$ tries Right and $\pi_C$ tries Up. Using the slip rule, list the cells the walker can end up in after each action and the probability of each. Which action can drop it into a hole? Does this explain the difference between your two estimates of V(1)?

   *your answer here*

3. Compare the mean episode lengths of the two policies. What does $\pi_C$ do that makes its episodes so much longer, and what does this mean for the cost of evaluating it?

   *your answer here*

4. Under $\pi_D$ some cells are visited rarely. In how many of your 1000 runs did cell 14 receive no estimate at all? What would you tell someone who used such an estimate to make a decision?

   *your answer here*

5. In the corridor, every-visit MC was visibly biased when only a few episodes were used. Is that the case in Table 16? Why might every-visit behave differently here, where every visit in an episode shares the same return?

   *your answer here*

---
## Part 6: Summary and Reflection

### Task 7: Summary and Reflection  *(15 min)*

**(a)** Table 17: Summary of concepts

| Concept | What it measures or does | How it was obtained in this lab |
|---|---|---|
| Return $G_t$ | | |
| Visit count N(s) | | |
| First-visit estimate | | |
| Every-visit estimate | | |
| $V_\pi$ from dynamic programming | | |

**(b)** In no more than 150 words, explain what Monte Carlo prediction offers compared with the dynamic programming of Labs 4 to 6, and name two limitations, one of which you observed in your own results (quote the number). State what would have to be added to move from estimating a policy to improving it.

*your answer here (max 150 words)*